# 02 - Inference

`InferenceEngine` is shared by the CLI, the Tk GUI and the HTTP API, so the
text you get here is the text they produce. The KV cache and the sampling
parameters are identical across all four front ends.

In [ ]:
import sys
from pathlib import Path

# Works whether TinyLLM is pip-installed or merely cloned next to this notebook.
def _repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "tinyllm" / "__init__.py").exists():
            return candidate
    return start

ROOT = _repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import tinyllm
print("tinyllm", tinyllm.__version__, "from", Path(tinyllm.__file__).parent)

In [ ]:
from tinyllm.features.inference.service import GenerationRequest, InferenceEngine
from tinyllm.core.paths import RUNS_DIR

candidates = sorted(RUNS_DIR.glob("*/checkpoint.pt"))
print("checkpoints found:")
for path in candidates:
    print("  ", path)

In [ ]:
CHECKPOINT = None                     # set to a path above to override
CHECKPOINT = CHECKPOINT or "models/runs/gpt-shakespeare/checkpoint.pt"

engine = InferenceEngine(device="auto")
if Path(CHECKPOINT).exists():
    engine.load(CHECKPOINT)
elif candidates:
    engine.load(candidates[-1])
else:
    print("no checkpoint yet - run 01_train.ipynb, or: python -m tinyllm train")

## One-shot generation

In [ ]:
request = GenerationRequest(
    prompt="ROMEO:",
    max_new_tokens=120,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
)
print(request.prompt, end="")

for piece in engine.stream(request):
    print(piece, end="", flush=True)
print()

## Streaming with and without the KV cache

Decoding with the cache appends one token per step. Turning it off recomputes
the whole prefix every step: slower, but a useful correctness check - both
paths must produce the same text.

In [ ]:
import time

for use_cache in (True, False):
    start = time.time()
    text = "".join(engine.stream(GenerationRequest(
        prompt="ROMEO:", max_new_tokens=40, use_cache=use_cache)))
    print(f"use_cache={use_cache!s:<5} {time.time() - start:6.2f}s  {text[:70]}...")

## Sampling parameters

In [ ]:
for temperature in (0.1, 0.7, 1.2):
    text = "".join(engine.stream(GenerationRequest(
        prompt="KING RICHARD II:", max_new_tokens=30, temperature=temperature)))
    print(f"T={temperature}: {text}")